In [3]:
# scrape_bonviveur_ingredients_general.py
# Requisits: pip install requests beautifulsoup4

import csv, re, time, random, json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE = "https://www.bonviveur.es"
START = "https://www.bonviveur.es/recetas/tag/cocina-catalana/"
HEADERS = {"User-Agent": "Mozilla/5.0 (SBCon/Blanca)"}
PAUSE = (0.8, 1.4)

def get_html(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    time.sleep(random.uniform(*PAUSE))
    return r.text

# ----------- llistat amb paginació ----------
def collect_links(list_url: str):
    links, seen = [], set()
    url = list_url
    while url:
        html = get_html(url)
        soup = BeautifulSoup(html, "html.parser")
        found = []
        for a in soup.select("a[href]"):
            href = a.get("href") or ""
            if "/recetas/" in href:
                full = urljoin(BASE, href)
                path = urlparse(full).path
                if "/recetas/lista/" in path:
                    continue
                if full not in seen:
                    seen.add(full)
                    links.append(full)
                    found.append(full)
        print(f"[Llista] +{len(found)} (acumulat {len(links)})")
        nxt = soup.select_one("a[rel='next']")
        url = urljoin(BASE, nxt["href"]) if nxt and nxt.get("href") else None
    return links

# ----------- utilitats d'extracció ----------
def text_of(sel: str, soup: BeautifulSoup) -> str:
    el = soup.select_one(sel)
    return el.get_text(strip=True) if el else ""

def extract_label(page_text: str, label: str) -> str:
    m = re.search(rf"{label}\s*:\s*([^\n\r|]+)", page_text, flags=re.I)
    return m.group(1).strip() if m else ""

def map_orden_from_categoria(cat: str) -> str:
    c = (cat or "").strip().lower()
    if c == "plato principal": return "Principal"
    if c == "entrantes": return "Entrante"
    if c == "postres": return "Postre"
    return ""

def clean_ing(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip(" -•\t\u2022").strip()
    return s

# ---- NORMALITZACIÓ D’INGREDIENTS (sense quantitats) ----
UNIT_RE = r"(g|gr|gramos?|kg|kilos?|mg|ml|l|litros?|cc|cm3|cda?s?|cucharaditas?|cucharadas?|cdta?s?|tazas?|vasos?|unidad(?:es)?|uds?|hojas?|dientes?|filetes?|rodajas?|trozos?|piezas?|latas?|botes?)\b\.?"
FRACTIONS_RE = r"(?:\d+/\d+|[¼½¾⅓⅔⅛⅜⅝⅞])"
NUM_RE = r"\d+(?:[.,]\d+)?"
DROP_PHRASES = [
    r"\bal gusto\b", r"\ba temperatura ambiente\b", r"\baprox\.\b",
    r"\b(opcional)\b", r"\bpara decorar\b", r"\ba elección\b", r"\bal dente\b"
]
SIZE_TOKENS = r"(?:xs|s|m|l|xl|grande?s?|pequeñ[oa]s?|medianas?)\b"

def normalize_ingredient_line(line: str) -> str:
    s = line.lower()
    s = re.sub(r"\([^)]*\)", " ", s)
    for pat in DROP_PHRASES:
        s = re.sub(pat, " ", s, flags=re.I)
    s = re.sub(FRACTIONS_RE, " ", s, flags=re.I)
    s = re.sub(NUM_RE, " ", s)
    s = re.sub(rf"\b{UNIT_RE}", " ", s, flags=re.I)
    s = re.sub(rf"\b{SIZE_TOKENS}", " ", s, flags=re.I)
    s = re.sub(r"[×x\*·]", " ", s)
    s = re.sub(r"\s+de\s+", " de ", s)
    s = re.sub(r"^\s*(de|del|la|el)\s+", "", s)
    s = re.sub(r"[\s,;:\-]+", " ", s).strip(" .,-/").strip()
    return s

def normalize_ingredients(list_ing):
    out = []
    seen = set()
    for raw in list_ing:
        base = clean_ing(raw)
        if not base: continue
        parts = re.split(r"\s*/\s*|\s*;\s*", base)
        for p in parts:
            p2 = normalize_ingredient_line(p)
            if p2 and p2 not in seen:
                seen.add(p2)
                out.append(p2)
    return out

# 1) JSON-LD (schema.org/Recipe → recipeIngredient)
def extract_ingredients_jsonld(soup: BeautifulSoup):
    items = []
    for tag in soup.select('script[type="application/ld+json"]'):
        try:
            data = tag.string
            if not data: continue
            obj = json.loads(data)
        except Exception:
            continue
        stack = obj if isinstance(obj, list) else [obj]
        for x in stack:
            if isinstance(x, dict) and (x.get("@type") == "Recipe" or ("Recipe" in str(x.get("@type")))):
                ingr = x.get("recipeIngredient") or []
                items.extend([clean_ing(i) for i in ingr if isinstance(i, str)])
    return [i for i in items if i]

# 2) DOM (secció “Ingredientes” + llistes properes)
def extract_ingredients_dom(soup: BeautifulSoup):
    items = []
    for sel in [
        ".ingredients li", ".ingredientes li", ".lista-ingredientes li",
        "[id*='ingred'] li", "[class*='ingred'] li"
    ]:
        for li in soup.select(sel):
            txt = clean_ing(li.get_text(" ", strip=True))
            if txt: items.append(txt)
    if not items:
        for h in soup.find_all(["h2","h3","h4"]):
            if "ingred" in (h.get_text(" ", strip=True) or "").lower():
                nxt = h.find_next(["ul","ol"])
                if nxt:
                    for li in nxt.select("li"):
                        txt = clean_ing(li.get_text(" ", strip=True))
                        if txt: items.append(txt)
    # dedup
    seen, out = set(), []
    for i in items:
        if i not in seen:
            seen.add(i)
            out.append(i)
    return out

# ------------------- REGOLES GENERALS BASADES EN INGREDIENTS -------------------
TOK_SWEET = {"azucar","azúcar","harina","maicena","leche","mantequilla","nata","crema","miel","chocolate","almendra","piñones","vainilla","limon","limón","huevo","yema","clara"}
TOK_MEAT_R = {"ternera","cordero","cerdo","morcillo","panceta","jamon","jamón","cap i pota","morcilla","butifarra","botifarra"}
TOK_POULTRY = {"pollo","gallina","pavo","codorniz"}
TOK_FISH = {"bacalao","bacallà","rape","merluza","atún","sardina","anchoa","sepia","pulpo","calamar","marisco","gamba","langostino","pescado"}
TOK_LEGUME_VEG = {"alubia","alubias","garbanzo","garbanzos","lenteja","lentejas","haba","habas","espinaca","espinacas","col","coliflor","berenjena","calabacin","calabacín","pimiento","tomate","patata","cebolla","ajo"}
TOK_GRAIN_PASTA = {"arroz","arròs","galets","canelones","fideos","pasta","macarrones"}
TOK_COLD_METHODS = {"ensalada","esqueixada","empedrat","marinado","marinada","vinagreta","carpaccio","tartar","en escabeche"}
TOK_HOT_METHODS = {"horno","horneado","brasa","guiso","guisado","estofado","frito","frita","sofrito","sopa","caldo","hervido","rustido","asado"}

def tokens_from(name: str, ings: list[str], page_text: str) -> set:
    base = f"{name} {' '.join(ings)} {page_text}".lower()
    # normalitza accents mínim
    repl = {
        "á":"a","é":"e","í":"i","ó":"o","ú":"u","ü":"u","ñ":"n","ç":"c","à":"a","è":"e","ò":"o","ï":"i"
    }
    for k,v in repl.items():
        base = base.replace(k,v)
    toks = set(re.findall(r"[a-zà-ÿA-ZÀ-ß][a-zà-ÿA-ZÀ-ß]+", base))
    return toks

def family_from_tokens(toks: set) -> str:
    if toks & TOK_FISH: return "fish"
    if toks & TOK_MEAT_R: return "meat"
    if toks & TOK_POULTRY: return "poultry"
    if toks & TOK_SWEET: return "dessert"
    if toks & (TOK_LEGUME_VEG | TOK_GRAIN_PASTA): return "veggie"
    return "veggie"

def infer_orden_general(orden_current: str, toks: set) -> str:
    if orden_current: return orden_current
    if toks & TOK_SWEET: return "Postre"
    if (toks & TOK_COLD_METHODS) and not (toks & (TOK_MEAT_R | TOK_POULTRY)):
        return "Entrante"
    return "Principal"

def infer_temperatura_general(toks: set) -> str:
    if toks & TOK_COLD_METHODS: return "Fred"
    if toks & TOK_HOT_METHODS: return "Calent"
    return ""  # desconegut

def infer_formalitat_general(toks: set, ing_count: int) -> str:
    # regla: proteïna cara o elaboració → Formal, dolços/familiars → Familiar, plats populars/cassolans → Tradicional
    if ("suquet" in toks or "zarzuela" in toks) or (("rape" in toks or "sepia" in toks) and ing_count >= 10):
        return "Formal"
    if toks & TOK_SWEET or "canelones" in toks or "escudella" in toks:
        return "Familiar"
    if ("cap" in toks and "pota" in toks) or "butifarra" in toks or "trinxat" in toks or "samfaina" in toks or "escalivada" in toks:
        return "Tradicional"
    if ing_count <= 6:
        return "Informal"
    return ""

def infer_compatibilidad_general(fam: str) -> str:
    if fam == "fish": return "vino blanco seco / cava brut"
    if fam == "meat": return "vino tinto crianza"
    if fam == "poultry": return "tinto joven / crianza suave"
    if fam == "dessert": return "moscatel / vino dulce / ratafia"
    return "blanco joven / rosado"

def infer_precio_general(fam: str, ing_count: int) -> str:
    # banda orientativa segons família + número d’ingredients
    base = {
        "dessert": 5.0,
        "veggie": 9.0,
        "poultry": 13.0,
        "meat": 15.0,
        "fish": 17.0,
    }.get(fam, 12.0)
    # ajust lleu per ingredients (més ingredients → més car/elaborat)
    adj = 0.0
    if ing_count >= 12: adj = 3.0
    elif ing_count >= 8: adj = 2.0
    elif ing_count >= 5: adj = 1.0
    return f"{base + adj:.1f}"

def infer_procedencia_general(tipo_cocina: str) -> str:
    # si BonViveur no la dona, usa "catalana" perquè venim d’un tag de 'cocina catalana'
    if tipo_cocina: return tipo_cocina
    return "catalana"

# ------------------- PARSE + ENRIQUEIX -------------------
def parse_recipe(url: str) -> dict:
    html = get_html(url)
    soup = BeautifulSoup(html, "html.parser")

    nombre = text_of("h1", soup) or (soup.select_one('meta[property="og:title"]') or {}).get("content","")
    bad = (nombre or "").lower()
    if any(k in bad for k in ["recetas de cocina", "las mejores recetas", "todas las recetas"]):
        return {}

    page_text = soup.get_text("\n", strip=True)
    dificultad   = extract_label(page_text, "Dificultad")
    tipo_cocina  = extract_label(page_text, "Tipo de cocina")
    categoria    = extract_label(page_text, "Categoría")
    orden_base   = map_orden_from_categoria(categoria)

    # ingredients: JSON-LD → DOM; normalitza
    ingredientes = extract_ingredients_jsonld(soup) or extract_ingredients_dom(soup)
    ingredientes_norm = normalize_ingredients(ingredientes)
    ing_count = len(ingredientes_norm)

    # --- INFERÈNCIA GENERAL EN BASE A INGREDIENTS + TEXT ---
    toks = tokens_from(nombre or "", ingredientes_norm, page_text)
    fam = family_from_tokens(toks)

    orden        = infer_orden_general(orden_base, toks)
    temperatura  = infer_temperatura_general(toks)
    formalitat   = infer_formalitat_general(toks, ing_count)
    compat       = infer_compatibilidad_general(fam)
    preu         = infer_precio_general(fam, ing_count)
    procedencia  = infer_procedencia_general(tipo_cocina)

    return {
        "nombre": (nombre or "").strip(),
        "Orden": orden,
        "compatibilidad": compat,
        "temperatura": temperatura,
        "procedencia plato": procedencia,
        "complejidad": (dificultad or "").strip(),
        "precio de venta": preu,
        "formalitat": formalitat,
        "ingredientes": " | ".join(ingredientes_norm) if ingredientes_norm else "",
        "url": url
    }

# ------------------ main ------------------
def main():
    print("[Inici] Recollint enllaços…")
    links = collect_links(START)
    print(f"[Total enllaços trobats] {len(links)}")

    rows = []
    for i, url in enumerate(links, 1):
        try:
            data = parse_recipe(url)
            if data.get("nombre"):
                rows.append(data)
                print(f"{i:03d}/{len(links)} {data['nombre']}")
        except Exception as e:
            print(f"[ERROR] {url} → {e}")

    fields = [
        "nombre","Orden","compatibilidad","temperatura",
        "procedencia plato","complejidad",
        "precio de venta","formalitat","ingredientes","url"
    ]
    with open("plats_catalans_bonviveur.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fields})

    print(f"\n[Fet] plats_catalans_bonviveur.csv ({len(rows)} files)")

if __name__ == "__main__":
    main()


[Inici] Recollint enllaços…
[Llista] +51 (acumulat 51)
[Total enllaços trobats] 51
002/51 Crema catalana
003/51 Calçots al horno
004/51 Fricandó de ternera
005/51 Pan con tomate
006/51 Escalivada
007/51 Escudella catalana
008/51 Coca de escalivada
009/51 Patatas de Olot
010/51 Salsa para calçots
011/51 Pollo con gambas
012/51 Patatas enmascaradas
013/51 Calçots a la brasa
014/51 Olla aranesa
015/51 Coca de Montserrat
016/51 Crema catalana en Thermomix
017/51 Sopa de Navidad
018/51 Panellets
019/51 Bacalao con samfaina
020/51 Pa de pessic
021/51 Pollo rustido
022/51 Xuxos o pepitos de crema
023/51 Mel i mató
024/51 Bacalao a la llauna
025/51 Pollo con ciruelas
026/51 Esqueixada de bacalao
027/51 Coca de Llavaneras
028/51 Coca de vidre
029/51 Cap i pota
030/51 Trinxat de la Cerdanya
031/51 Canelones de San Esteban
032/51 Carquiñoles
033/51 Bombas de patata con carne picada
034/51 Espinacas a la catalana
035/51 Coca de forner o coca de pan dulce
036/51 Sopa de galets
037/51 Coca de llardo

In [2]:
!pip install bs4

  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 3.7 MB/s eta 0:00:00
Using cached soupsieve-2.8-py3-none-any.whl (36 kB)
